In this notebook I will create a training table equivalent to ml_model_run_details

# Define Library

In [1]:
# %% [markdown]
# # Jupyter Notebook Loading Header
#
# This is a custom loading header for Jupyter Notebooks in Visual Studio Code.
# It includes common imports and settings to get you started quickly.
# %% [markdown]
## Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery
from google.cloud import storage
import os
import tempfile
import time
from datetime import datetime
import uuid
import joblib
import uuid

import gcsfs
import duckdb as dd
import pickle
import joblib
from typing import Union
import io
path = r'C:\Users\Dwaipayan\AppData\Roaming\gcloud\application_default_credentials.json'
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = path
client = bigquery.Client(project='prj-prod-dataplatform')
os.environ["GOOGLE_CLOUD_PROJECT"] = "prj-prod-dataplatform"

# %% [markdown]
## Configure Settings
# Set options or configurations as needed
pd.set_option('display.max_columns', None)
pd.set_option("Display.max_rows", 100)

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\auth\_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


### Function

#### expand_calc_features

In [2]:
import pandas as pd
import json

def expand_calc_features(df):
    """
    Expand the calcFeatures JSON column into separate columns and return the complete DataFrame.

    Parameters:
    df (pd.DataFrame): Input DataFrame with calcFeatures column containing JSON data

    Returns:
    pd.DataFrame: Expanded DataFrame with all original columns plus JSON features as separate columns
    """

    # Make a copy to avoid modifying the original DataFrame
    df_expanded = df.copy()

    # Parse the calcFeatures JSON column
    calc_features_list = []

    for idx, calc_features_str in enumerate(df['calcFeatures']):
        try:
            # Parse the JSON string
            features_dict = json.loads(calc_features_str.replace("'", '"'))  # Replace single quotes with double quotes for valid JSON
            calc_features_list.append(features_dict)
        except (json.JSONDecodeError, AttributeError) as e:
            # If parsing fails, create an empty dict and print warning
            print(f"Warning: Could not parse calcFeatures at index {idx}: {e}")
            calc_features_list.append({})

    # Create DataFrame from the parsed JSON data
    calc_features_df = pd.DataFrame(calc_features_list)

    # Add prefix to JSON-derived columns to avoid conflicts
    calc_features_df = calc_features_df.add_prefix('calc_')

    # Reset index to ensure proper alignment
    df_expanded = df_expanded.reset_index(drop=True)
    calc_features_df = calc_features_df.reset_index(drop=True)

    # Combine original DataFrame with expanded calcFeatures
    result_df = pd.concat([df_expanded, calc_features_df], axis=1)

    return result_df


#### expand_calc_features_robust

In [3]:
import pandas as pd
import json

def expand_calc_features_robust(df):
    """
    Expand the calcFeatures JSON column into separate columns with better error handling.

    Parameters:
    df (pd.DataFrame): Input DataFrame with calcFeatures column containing JSON data

    Returns:
    pd.DataFrame: Expanded DataFrame with all original columns plus JSON features as separate columns
    """

    # Make a copy to avoid modifying the original DataFrame
    df_expanded = df.copy()

    # Parse the calcFeatures JSON column
    calc_features_data = []

    for idx, row in df.iterrows():
        calc_features_str = row['calcFeatures']

        if pd.isna(calc_features_str) or calc_features_str == '':
            calc_features_data.append({})
            continue

        try:
            # Clean the string and parse JSON
            cleaned_str = calc_features_str.replace("'", '"').replace('None', 'null').replace('True', 'true').replace('False', 'false')
            features_dict = json.loads(cleaned_str)
            calc_features_data.append(features_dict)
        except Exception as e:
            print(f"Warning: Could not parse calcFeatures at index {idx}: {e}")
            print(f"Problematic string: {calc_features_str[:100]}...")  # Print first 100 chars
            calc_features_data.append({})

    # Create DataFrame from the parsed JSON data
    calc_features_df = pd.DataFrame(calc_features_data)

    # Add prefix to JSON-derived columns to avoid conflicts with existing columns
    calc_features_df = calc_features_df.add_prefix('feat_')

    # Combine DataFrames
    result_df = pd.concat([df_expanded, calc_features_df], axis=1)

    print(f"Original DataFrame shape: {df.shape}")
    print(f"Expanded DataFrame shape: {result_df.shape}")
    print(f"Added {len(calc_features_df.columns)} new columns from calcFeatures")

    return result_df

# Table Name

In [4]:
table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"

# transform_datav2

In [5]:
import pandas as pd
import json
import uuid
from datetime import datetime
from typing import List

def transform_datav2(
    d1: pd.DataFrame, 
    feature_column: List[str], 
    a: str = 'demo_score', 
    modelDisplayName: str = 'Cash_beta_trench1_Demo_backscore', 
    tc: str = "", 
    subscription_name: str = 'sil_march 25 models'
) -> pd.DataFrame:
    """
    Transforms input data into a structured format suitable for model scoring output.

    Parameters:
    - d1 (pd.DataFrame): Input DataFrame containing raw data.
    - feature_column (List[str]): List of column names to include in the 'calcFeature' JSON.
    - a (str): Column name containing the prediction score. Default is 'demo_score'.
    - modelDisplayName (str): Name of the model used for scoring.
    - tc (str): Trench category (optional).
    - do (str): Device operating system. Default is 'android'.
    - subscription_name (str): Name of the subscription or model group.

    Returns:
    - pd.DataFrame: Transformed DataFrame with structured output.
    """

    # Make a copy of the input DataFrame to avoid modifying the original
    df = d1.copy()
    
    # Initialize an empty list to store transformed rows
    output_data = []
    
    # Iterate over each row in the DataFrame
    for _, row in df.iterrows():
        # Initialize dictionary to hold feature values
        calc_feature = {}
        
        # Loop through each feature column and extract its value from the row
        for col in feature_column:
            if col in row and pd.notna(row[col]):
                # Convert datetime values to ISO format strings
                if isinstance(row[col], pd.Timestamp):
                    calc_feature[col] = row[col].isoformat()
                else:
                    calc_feature[col] = row[col]
        
        # Get the current timestamp for start_time, end_time, and publish_time
        current_time = datetime.now().isoformat()
        
        # Construct the output row dictionary with required fields
        output_row = {
            "customerId": row['customer_id'],  # Unique customer identifier
            "digitalLoanAccountId": row['digitalLoanAccountId'],  # Loan account ID
            "crifApplicationId": str(uuid.uuid4()),  # Random UUID for application ID
            "prediction": row.get(a, 0),  # Prediction score from specified column
            "start_time": current_time,  # Timestamp when processing starts
            "end_time": current_time,    # Timestamp when processing ends
            "modelDisplayName": modelDisplayName,  # Name of the model used
            "modelVersionId": "v2",  # Static model version
            "calcFeature": json.dumps(calc_feature, default=str),  # Features as JSON string
            "subscription_name": subscription_name,  # Subscription name
            "message_id": str(uuid.uuid4()),  # Random UUID for message ID
            "publish_time": current_time,  # Timestamp when message is published
            "attributes": "{}",  # Placeholder for additional attributes
            "trenchCategory": tc,  # Optional trench category
            "deviceOs": row['osType'],  # Device operating system
            "Data_selection": row['Data_selection'],  # Data selection
            "Application_date": row['application_date'],
        }
        
        # Append the transformed row to the output list
        output_data.append(output_row)
    
    # Convert the list of dictionaries to a DataFrame
    output_df = pd.DataFrame(output_data)
    
    # Return the transformed DataFrame
    return output_df


# SIL V2

*   Sil-Alpha-CIC-SIL-Model: worktable_data_analysis.sil_alpha_cic_all_applied_backscored_20240901_20250930
*   Sil-Alpha-StackingModel: worktable_data_analysis.sil_alpha_applied_loans_backscored_20240901_20251013_option3
*   Sil-Beta-AppsScoreModel:
*   New Applicants (T1, T2): risk_mart.applied_sil_trench1_trench2_loan_jan2024_30sep2025_app_scored, risk_mart.applied_sil_new_applicants_loan_oct0125_oct1325_app_scored
*   Repeat Applicants (T3): risk_mart.b_score_model_applied_loans_sil_20240101_20250930_app_scored, risk_mart.applied_sil_repeat_applicants_loan_oct0125_oct1325_app_scored
*   Sil-Beta-DemoScoreModel: worktable_data_analysis.sil_beta_demo_all_applied_backscored_20240801_20251015
*   Sil-Beta-StackScoreModel: worktable_data_analysis.sil_beta_applied_loans_backscored_20240801_20251013_option3m

##### 'Alpha - CIC-SIL-Model'

##### Trench 1

In [ ]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='cic_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 1'
;
"""

client.query(sq)

In [ ]:
sq = """ 
select 
r.customerId customer_id,
r.digitalLoanAccountId, 
r.c_cic_score ,
r.ScoreRange,
       ln_loan_level_user_type, flg_zero_non_granted_ever,
       flg_zero_granted_ever,
       Personal_Loans_granted_contracts_amt_24M,
       granted_contracts_cnt_6M, total_overdue_granted_contracts,
       has_ever_been_overdue, cnt_nongranted_contracts_3M,
       cnt_active_contracts, max_amt_granted_24M,
       tot_active_contracts_util, days_since_last_closed,
       vel_contract_nongranted_cnt_6on12,
       vel_contract_granted_amt_6on12,
       vel_contract_closed_amt_3on12,
case 
  when lower(r.ln_os_type) like '%andro%' then 'android'
  when lower(r.ln_os_type) like '%os%' then 'ios'
  else 'ios' end  osType,
 date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
 case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-09-01' and '2025-01-31' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-09-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection 
from worktable_data_analysis.sil_alpha_cic_all_applied_backscored_20240901_20250930 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.trench_category = 'Trench 1'
and r.c_cic_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

In [ ]:
data.columns

In [ ]:
data.head()

In [ ]:
feature_column = ['ScoreRange',
       'ln_loan_level_user_type', 'flg_zero_non_granted_ever',
       'flg_zero_granted_ever', 'Personal_Loans_granted_contracts_amt_24M',
       'granted_contracts_cnt_6M', 'total_overdue_granted_contracts',
       'has_ever_been_overdue', 'cnt_nongranted_contracts_3M',
       'cnt_active_contracts', 'max_amt_granted_24M',
       'tot_active_contracts_util', 'days_since_last_closed',
       'vel_contract_nongranted_cnt_6on12', 'vel_contract_granted_amt_6on12',
       'vel_contract_closed_amt_3on12',]

dfd = transform_datav2(data, feature_column, a='c_cic_score', modelDisplayName='cic_model_sil', tc='Trench 1', subscription_name = 'Cash November 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

##### Trench 2

In [ ]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='cic_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 2'
;"""

client.query(sq)

In [ ]:
sq = """ 
select 
r.customerId customer_id,
r.digitalLoanAccountId, 
r.c_cic_score ,
r.ScoreRange,
       ln_loan_level_user_type, flg_zero_non_granted_ever,
       flg_zero_granted_ever,
       Personal_Loans_granted_contracts_amt_24M,
       granted_contracts_cnt_6M, total_overdue_granted_contracts,
       has_ever_been_overdue, cnt_nongranted_contracts_3M,
       cnt_active_contracts, max_amt_granted_24M,
       tot_active_contracts_util, days_since_last_closed,
       vel_contract_nongranted_cnt_6on12,
       vel_contract_granted_amt_6on12,
       vel_contract_closed_amt_3on12,
case 
  when lower(r.ln_os_type) like '%andro%' then 'android'
  when lower(r.ln_os_type) like '%os%' then 'ios'
  else 'ios' end  osType,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
 case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-09-01' and '2025-01-31' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-09-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection 
from worktable_data_analysis.sil_alpha_cic_all_applied_backscored_20240901_20250930 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.trench_category = 'Trench 2'
and r.c_cic_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-11-17'
 ;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

In [ ]:
data.columns

In [ ]:
data.head()

In [ ]:
feature_column = ['ScoreRange',
       'ln_loan_level_user_type', 'flg_zero_non_granted_ever',
       'flg_zero_granted_ever', 'Personal_Loans_granted_contracts_amt_24M',
       'granted_contracts_cnt_6M', 'total_overdue_granted_contracts',
       'has_ever_been_overdue', 'cnt_nongranted_contracts_3M',
       'cnt_active_contracts', 'max_amt_granted_24M',
       'tot_active_contracts_util', 'days_since_last_closed',
       'vel_contract_nongranted_cnt_6on12', 'vel_contract_granted_amt_6on12',
       'vel_contract_closed_amt_3on12',]

dfd = transform_datav2(data, feature_column, a='c_cic_score', modelDisplayName='cic_model_sil', tc='Trench 2', subscription_name = 'Cash November 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

##### Trench 3

In [ ]:
sq = """
delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='cic_model_sil'
and modelVersionId = 'v1'
and trenchCategory = 'Trench 3'
;
"""

client.query(sq)

In [ ]:
sq = """ 
select 
r.customerId customer_id,
r.digitalLoanAccountId, 
r.c_cic_score ,
r.ScoreRange,
       ln_loan_level_user_type, flg_zero_non_granted_ever,
       flg_zero_granted_ever,
       Personal_Loans_granted_contracts_amt_24M,
       granted_contracts_cnt_6M, total_overdue_granted_contracts,
       has_ever_been_overdue, cnt_nongranted_contracts_3M,
       cnt_active_contracts, max_amt_granted_24M,
       tot_active_contracts_util, days_since_last_closed,
       vel_contract_nongranted_cnt_6on12,
       vel_contract_granted_amt_6on12,
       vel_contract_closed_amt_3on12,
case 
  when lower(r.ln_os_type) like '%andro%' then 'android'
  when lower(r.ln_os_type) like '%os%' then 'ios'
  else 'ios' end  osType,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
 case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-09-01' and '2025-01-31' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-09-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection 
from worktable_data_analysis.sil_alpha_cic_all_applied_backscored_20240901_20250930 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.trench_category = 'Trench 3'
and r.c_cic_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

In [ ]:
data.columns

In [ ]:
data.head()

In [ ]:
feature_column = ['ScoreRange',
       'ln_loan_level_user_type', 'flg_zero_non_granted_ever',
       'flg_zero_granted_ever', 'Personal_Loans_granted_contracts_amt_24M',
       'granted_contracts_cnt_6M', 'total_overdue_granted_contracts',
       'has_ever_been_overdue', 'cnt_nongranted_contracts_3M',
       'cnt_active_contracts', 'max_amt_granted_24M',
       'tot_active_contracts_util', 'days_since_last_closed',
       'vel_contract_nongranted_cnt_6on12', 'vel_contract_granted_amt_6on12',
       'vel_contract_closed_amt_3on12',]

dfd = transform_datav2(data, feature_column, a='c_cic_score', modelDisplayName='cic_model_sil', tc='Trench 3', subscription_name = 'Cash November 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

##### Alpha Sil Stack Model 

##### Trench 1

In [ ]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='alpha_stack_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 1'
;"""

client.query(sq)

In [ ]:
sq = """ 
select 
lmt.customerId customer_id,
r.digitalLoanAccountId,  
r.sa_stack_score ,
r.sb_demo_score,
r.apps_score,
r.s_credo_score,
r.sa_cic_score,
case 
  when lower(r.ln_os_type) like '%andro%' then 'android'
  when lower(r.ln_os_type) like '%os%' then 'ios'
  else 'ios' end  osType,
ln_loan_type,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-09-01' and '2025-02-28' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-09-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from worktable_data_analysis.sil_alpha_applied_loans_backscored_20240901_20251013_option3 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.trench_category = 'Trench 1'
and sa_stack_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


In [ ]:
data.columns

In [ ]:
data.head()

In [ ]:
feature_column = ['sb_demo_score', 'apps_score', 's_credo_score', 'sa_cic_score',]

dfd = transform_datav2(data, feature_column, a='sa_stack_score', modelDisplayName='alpha_stack_model_sil', tc='Trench 1', subscription_name = 'Cash November 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

##### Trench 2

In [ ]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='alpha_stack_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 2'
;"""

client.query(sq)

In [ ]:
sq = """ 
select 
lmt.customerId customer_id,
r.digitalLoanAccountId,  
r.sa_stack_score ,
r.sb_demo_score,
r.apps_score,
r.s_credo_score,
r.sa_cic_score,
case 
  when lower(r.ln_os_type) like '%andro%' then 'android'
  when lower(r.ln_os_type) like '%os%' then 'ios'
  else 'ios' end  osType,
ln_loan_type,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-09-01' and '2025-02-28' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-09-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from worktable_data_analysis.sil_alpha_applied_loans_backscored_20240901_20251013_option3 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.trench_category = 'Trench 2'
and sa_stack_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


In [ ]:
data.columns

In [ ]:
data.head()

In [ ]:
feature_column = ['sb_demo_score', 'apps_score', 's_credo_score', 'sa_cic_score',]

dfd = transform_datav2(data, feature_column, a='sa_stack_score', modelDisplayName='alpha_stack_model_sil', tc='Trench 2', subscription_name = 'Cash November 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

##### Trench 3

In [ ]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='alpha_stack_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 3'
;"""

client.query(sq)

In [ ]:
sq = """ 
select 
lmt.customerId customer_id,
r.digitalLoanAccountId,  
r.sa_stack_score ,
r.sb_demo_score,
r.apps_score,
r.s_credo_score,
r.sa_cic_score,
case 
  when lower(r.ln_os_type) like '%andro%' then 'android'
  when lower(r.ln_os_type) like '%os%' then 'ios'
  else 'ios' end  osType,
ln_loan_type,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-09-01' and '2025-02-28' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-09-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from worktable_data_analysis.sil_alpha_applied_loans_backscored_20240901_20251013_option3 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.trench_category = 'Trench 3'
and sa_stack_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


In [ ]:
data.columns

In [ ]:
data.head()

In [ ]:
feature_column = ['sb_demo_score', 'apps_score', 's_credo_score', 'sa_cic_score',]

dfd = transform_datav2(data, feature_column, a='sa_stack_score', modelDisplayName='alpha_stack_model_sil', tc='Trench 3', subscription_name = 'Cash November 25 Models')
print(f"The shape of the transformed dataframe is:\t {dfd.shape}") 
dfd.head()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

##### Beta SIL STACK Score Model

##### Trench 1

In [ ]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='beta_stack_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 1'
;"""

client.query(sq)

In [ ]:
sq = """ 
select 
lmt.customerId customer_id,
r.digitalLoanAccountId,  
r.sb_stack_score ,
r.sb_demo_score,
r.apps_score,
r.s_credo_score,
case 
  when lower(r.ln_os_type) like '%andro%' then 'android'
  when lower(r.ln_os_type) like '%os%' then 'ios'
  else 'ios' end  osType,
ln_loan_type,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-08-01' and '2025-02-28' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-08-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from prj-prod-dataplatform.worktable_data_analysis.sil_beta_applied_loans_backscored_20240801_20251013_option3 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.trench_category = 'Trench 1'
and sb_stack_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


In [ ]:
data.columns

In [ ]:
data.head()

In [ ]:
feature_column = ['sb_demo_score', 'apps_score', 's_credo_score']

dfd = transform_datav2(data, feature_column, a='sb_stack_score', modelDisplayName='beta_stack_model_sil', tc='Trench 1', subscription_name = 'Cash November 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

##### Trench 2

In [ ]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='beta_stack_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 2'
;"""

client.query(sq)

In [ ]:
sq = """ 
select 
lmt.customerId customer_id,
r.digitalLoanAccountId,  
r.sb_stack_score ,
r.sb_demo_score,
r.apps_score,
r.s_credo_score,
case 
  when lower(r.ln_os_type) like '%andro%' then 'android'
  when lower(r.ln_os_type) like '%os%' then 'ios'
  else 'ios' end  osType,
ln_loan_type,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-08-01' and '2025-02-28' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-08-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from prj-prod-dataplatform.worktable_data_analysis.sil_beta_applied_loans_backscored_20240801_20251013_option3 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.trench_category = 'Trench 2'
and sb_stack_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


In [ ]:
data.columns

In [ ]:
data.head()

In [ ]:
feature_column = ['sb_demo_score', 'apps_score', 's_credo_score']

dfd = transform_datav2(data, feature_column, a='sb_stack_score', modelDisplayName='beta_stack_model_sil', tc='Trench 2', subscription_name = 'Cash November 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

##### Trench 3

In [ ]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='beta_stack_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 3'
;"""

client.query(sq)

In [ ]:
sq = """ 
select 
lmt.customerId customer_id,
r.digitalLoanAccountId,  
r.sb_stack_score ,
r.sb_demo_score,
r.apps_score,
r.s_credo_score,
case 
  when lower(r.ln_os_type) like '%andro%' then 'android'
  when lower(r.ln_os_type) like '%os%' then 'ios'
  else 'ios' end  osType,
ln_loan_type,
date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime))
        between '2024-08-01' and '2025-02-28' then 'Dev_Train'
        when date(if(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2024-08-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from prj-prod-dataplatform.worktable_data_analysis.sil_beta_applied_loans_backscored_20240801_20251013_option3 r
left join `risk_credit_mis.loan_master_table` lmt on lmt.digitalLoanAccountId = r.digitalLoanAccountId
where r.trench_category = 'Trench 3'
and sb_stack_score is not null
and date(IF(lmt.new_loan_type = 'Flex-up', lmt.startApplyDateTime, lmt.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


In [ ]:
data.columns

In [ ]:
data.head()

In [ ]:
feature_column = ['sb_demo_score', 'apps_score', 's_credo_score']

dfd = transform_datav2(data, feature_column, a='sb_stack_score', modelDisplayName='beta_stack_model_sil', tc='Trench 3', subscription_name = 'Cash November 25 Models') 
print(f"The shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

##### Beta Sil App Score

##### Trench 1

In [ ]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='apps_score_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 1'
;"""

client.query(sq)

In [ ]:
sq = """ 
select 
distinct 
loanmaster.customerId customer_id,
r.digitalLoanAccountId,
r.apps_score,
r.lr_score ml_score,
r.dl_score dl_score,
app_cnt_payday_ever,
app_cnt_finance_ever,
app_cnt_competitors_sil_ever,
app_cnt_competitors_ever,
app_cnt_finance_365d,
app_cnt_absence_tag_365d,
app_cnt_competitors_sil_365d,
app_cnt_finance_7d,
app_cnt_rated_for_3plus_ever,
app_cnt_payday_7d,
    case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
    when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
    when lower(loanmaster.deviceType) like '%andro%' then 'android'
    else 'ios' end osType,
date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime))
        between '2024-08-01' and '2025-01-31' then 'Dev_Train'
        when date(if(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2024-08-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from  risk_mart.applied_sil_new_applicants_loan_20241001_20251124_app_scored r
left join risk_credit_mis.loan_master_table loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where apps_score is not null
and trench_category = 'Trench 1'
and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


In [ ]:
data.columns

In [ ]:
data.head()

In [ ]:
feature_column = ['ml_score', 'dl_score', 'app_cnt_payday_ever', 'app_cnt_finance_ever',
       'app_cnt_competitors_sil_ever', 'app_cnt_competitors_ever',
       'app_cnt_finance_365d', 'app_cnt_absence_tag_365d',
       'app_cnt_competitors_sil_365d', 'app_cnt_finance_7d',
       'app_cnt_rated_for_3plus_ever', 'app_cnt_payday_7d',]

dfd = transform_datav2(data, feature_column, a='apps_score', modelDisplayName='apps_score_model_sil', tc='Trench 1', subscription_name = 'Cash November 25 Models')
print(f"the shape of the transformed dataframe is:\t {dfd.shape}") 
dfd.head()

In [ ]:

result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

##### Trench 2

In [ ]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='apps_score_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 2'
;"""

client.query(sq)

In [ ]:
sq = """ 
select 
distinct 
loanmaster.customerId customer_id,
r.digitalLoanAccountId,
r.apps_score,
r.lr_score ml_score,
r.dl_score dl_score,
app_cnt_payday_ever,
app_cnt_finance_ever,
app_cnt_competitors_sil_ever,
app_cnt_competitors_ever,
app_cnt_finance_365d,
app_cnt_absence_tag_365d,
app_cnt_competitors_sil_365d,
app_cnt_finance_7d,
app_cnt_rated_for_3plus_ever,
app_cnt_payday_7d,
    case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
    when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
    when lower(loanmaster.deviceType) like '%andro%' then 'android'
    else 'ios' end osType,
date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime))
        between '2024-08-01' and '2025-01-31' then 'Dev_Train'
        when date(if(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2024-08-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from  risk_mart.applied_sil_new_applicants_loan_20241001_20251124_app_scored r
left join risk_credit_mis.loan_master_table loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where apps_score is not null
and trench_category = 'Trench 2'
and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


In [ ]:
data.columns

In [ ]:
data.head()

In [ ]:
feature_column = ['ml_score', 'dl_score', 'app_cnt_payday_ever', 'app_cnt_finance_ever',
       'app_cnt_competitors_sil_ever', 'app_cnt_competitors_ever',
       'app_cnt_finance_365d', 'app_cnt_absence_tag_365d',
       'app_cnt_competitors_sil_365d', 'app_cnt_finance_7d',
       'app_cnt_rated_for_3plus_ever', 'app_cnt_payday_7d',]

dfd = transform_datav2(data, feature_column, a='apps_score', modelDisplayName='apps_score_model_sil', tc='Trench 2', subscription_name = 'Cash November 25 Models') 
print(f"the shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

##### Trench 3

In [ ]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='apps_score_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 3'
;"""

client.query(sq)

In [ ]:
sq = """ 
select 
r.customerId customer_id,
r.digitalLoanAccountId,
r.apps_score,
r.cb_score ml_score,
r.dl_score dl_score,
app_cnt_productivity_ever,
app_cnt_rated_for_3plus_ever, 
app_cnt_books_and_reference_ever,
app_cnt_tools_ever, 
app_median_time_bw_installed_mins_3d,
app_median_time_bw_installed_mins_30d,
app_cnt_communication_ever, 
app_cnt_finance_90d,
app_cnt_absence_tag_180d, 
app_cnt_shopping_ever,
app_cnt_social_ever, 
app_cnt_driver_90d, 
app_cnt_payday_365d,
app_cnt_driver_365d, 
app_cnt_music_and_audio_ever,
app_cnt_finance_180d, 
app_cnt_art_and_design_ever,
app_cnt_gaming_90d, 
app_avg_time_bw_installed_mins_30d,
app_cnt_education_ever,
case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
    when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
    when lower(loanmaster.deviceType) like '%andro%' then 'android'
    else 'ios' end osType,
date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime))
        between '2024-08-01' and '2025-01-31' then 'Dev_Train'
        when date(if(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2024-08-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from risk_mart.applied_sil_repeat_applicants_loan_20241001_20251124_app_scored r
left join risk_credit_mis.loan_master_table loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where apps_score is not null
and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

In [ ]:
data.columns

In [ ]:
feature_column = ['ml_score', 'dl_score', 'app_cnt_productivity_ever', 'app_cnt_rated_for_3plus_ever',
       'app_cnt_books_and_reference_ever', 'app_cnt_tools_ever',
       'app_median_time_bw_installed_mins_3d',
       'app_median_time_bw_installed_mins_30d', 'app_cnt_communication_ever',
       'app_cnt_finance_90d', 'app_cnt_absence_tag_180d',
       'app_cnt_shopping_ever', 'app_cnt_social_ever', 'app_cnt_driver_90d',
       'app_cnt_payday_365d', 'app_cnt_driver_365d',
       'app_cnt_music_and_audio_ever', 'app_cnt_finance_180d',
       'app_cnt_art_and_design_ever', 'app_cnt_gaming_90d',
       'app_avg_time_bw_installed_mins_30d', 'app_cnt_education_ever',]

dfd = transform_datav2(data, feature_column, a='apps_score', modelDisplayName='apps_score_model_sil', tc='Trench 3', subscription_name = 'Cash November 25 Models') 
print(f"the shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

In [ ]:
dfd['customerId'] = pd.to_numeric(dfd['customerId'], errors='coerce')

In [ ]:
dfd.info()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

##### Beta SIL Demo Score

##### Trench 1

In [ ]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='beta_demo_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 1'
;"""

client.query(sq)

In [ ]:
sq = """select 
distinct 
loanmaster.customerId customer_id,
r.digitalLoanAccountId,
r.s_demo_score,
ln_vas_opted_flag,
ln_doc_type_rolled,
ln_industry_new_cat_bin,
ln_marital_status,
ln_age,
ln_education_level,
ln_cnt_dependents,
ln_ref2_type,
ln_loan_level_user_type,
ln_ref1_type,
ln_name_email_match_score,
ln_telconame,
ln_city_cat,
ln_brand_bin,
ln_apply_Is_Weekend,
r.ln_os_type osType,
date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime))
        between '2024-08-01' and '2025-01-31' then 'Dev_Train'
        when date(if(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2024-08-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from worktable_data_analysis.sil_beta_demo_all_applied_backscored_20240801_20251015 r
left join risk_credit_mis.loan_master_table loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where s_demo_score is not null
and trench_category = 'Trench 1'
and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

In [ ]:
data.columns

In [ ]:
feature_column = [ 'ln_vas_opted_flag', 'ln_doc_type_rolled', 'ln_industry_new_cat_bin',
       'ln_marital_status', 'ln_age', 'ln_education_level',
       'ln_cnt_dependents', 'ln_ref2_type', 'ln_loan_level_user_type',
       'ln_ref1_type', 'ln_name_email_match_score', 'ln_telconame',
       'ln_city_cat', 'ln_brand_bin', 'ln_apply_Is_Weekend', 'osType']

dfd = transform_datav2(data, feature_column, a='s_demo_score', modelDisplayName='beta_demo_model_sil', tc='Trench 1', subscription_name = 'Cash November 25 Models') 
print(f"the shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

In [ ]:
dfd['customerId'] = pd.to_numeric(dfd['customerId'], errors='coerce')

In [ ]:
dfd.info()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

##### Trench 2

In [ ]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='beta_demo_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 2'
;"""

client.query(sq)

In [ ]:
sq = """select 
distinct 
loanmaster.customerId customer_id,
r.digitalLoanAccountId,
r.s_demo_score,
ln_vas_opted_flag,
ln_doc_type_rolled,
ln_industry_new_cat_bin,
ln_marital_status,
ln_age,
ln_education_level,
ln_cnt_dependents,
ln_ref2_type,
ln_loan_level_user_type,
ln_ref1_type,
ln_name_email_match_score,
ln_telconame,
ln_city_cat,
ln_brand_bin,
ln_apply_Is_Weekend,
r.ln_os_type osType,
date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime))
        between '2024-08-01' and '2025-01-31' then 'Dev_Train'
        when date(if(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2024-08-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from worktable_data_analysis.sil_beta_demo_all_applied_backscored_20240801_20251015 r
left join risk_credit_mis.loan_master_table loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where s_demo_score is not null
and trench_category = 'Trench 2'
and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2025-11-17'
;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

In [ ]:
data.columns

In [ ]:
feature_column = [ 'ln_vas_opted_flag', 'ln_doc_type_rolled', 'ln_industry_new_cat_bin',
       'ln_marital_status', 'ln_age', 'ln_education_level',
       'ln_cnt_dependents', 'ln_ref2_type', 'ln_loan_level_user_type',
       'ln_ref1_type', 'ln_name_email_match_score', 'ln_telconame',
       'ln_city_cat', 'ln_brand_bin', 'ln_apply_Is_Weekend', 'osType']

dfd = transform_datav2(data, feature_column, a='s_demo_score', modelDisplayName='beta_demo_model_sil', tc='Trench 2', subscription_name = 'Cash November 25 Models') 
print(f"the shape of the transformed dataframe is:\t {dfd.shape}")
dfd.head()

In [ ]:
dfd['customerId'] = pd.to_numeric(dfd['customerId'], errors='coerce')

In [ ]:
dfd.info()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

##### Trench 3

In [ ]:
sq = """delete from prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName='beta_demo_model_sil'
and modelVersionId = 'v2'
and trenchCategory = 'Trench 3'
;"""

client.query(sq)

In [ ]:
sq = """select 
distinct 
loanmaster.customerId customer_id,
r.digitalLoanAccountId,
r.s_demo_score,
ln_vas_opted_flag,
ln_doc_type_rolled,
ln_industry_new_cat_bin,
ln_marital_status,
ln_age,
ln_education_level,
ln_cnt_dependents,
ln_ref2_type,
ln_loan_level_user_type,
ln_ref1_type,
ln_name_email_match_score,
ln_telconame,
ln_city_cat,
ln_brand_bin,
ln_apply_Is_Weekend,
r.ln_os_type osType,
date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) application_date,
case when date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime))
        between '2024-08-01' and '2025-01-31' then 'Dev_Train'
        when date(if(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2024-08-01' then 'Pre_Train'
                else 'Dev_Test' end as Data_selection
from worktable_data_analysis.sil_beta_demo_all_applied_backscored_20240801_20251015 r
left join risk_credit_mis.loan_master_table loanmaster
  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
where s_demo_score is not null
and trench_category = 'Trench 3'
and date(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime)) < '2025-11-17'
 ;
"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

In [ ]:
data.columns

In [ ]:
feature_column = [ 'ln_vas_opted_flag', 'ln_doc_type_rolled', 'ln_industry_new_cat_bin',
       'ln_marital_status', 'ln_age', 'ln_education_level',
       'ln_cnt_dependents', 'ln_ref2_type', 'ln_loan_level_user_type',
       'ln_ref1_type', 'ln_name_email_match_score', 'ln_telconame',
       'ln_city_cat', 'ln_brand_bin', 'ln_apply_Is_Weekend', 'osType']

dfd = transform_datav2(data, feature_column, a='s_demo_score', modelDisplayName='beta_demo_model_sil', tc='Trench 3', subscription_name = 'Cash November 25 Models') 
dfd.head()

In [ ]:
dfd['customerId'] = pd.to_numeric(dfd['customerId'], errors='coerce')

In [ ]:
dfd.info()

In [ ]:

result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

# 💎 Cash V2

## Ⓜ️ alpha_stack_model_cash

### ₸ Trench 1

#### ₫ Delete

In [6]:
sq = """
DELETE FROM `prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116`
WHERE modelDisplayName = 'Alpha-Cash-Stack-Model'
  AND modelVersionId = 'v2'
  AND trenchCategory = 'Trench 1';
"""

query_job = client.query(sq)
query_job.result()

print(f"Deleted {query_job.num_dml_affected_rows} rows.")

Deleted 0 rows.


In [13]:
sq = """
SELECT
  r.`customerId` as customer_id,
  r.`digitalLoanAccountId`,
  r.`ln_appln_submit_datetime` application_date,
  r.`ln_loan_type`,
  r.`ln_loan_level_user_type`,
  r.`trench_category`,
  r.`ln_os_type` as osType,
  r.`is_android`,
  r.`cb_demo_score` as demo_score,
  r.`apps_score` as app_score,
  r.`device_score` as device_score,
  r.`c_credo_score` as credo_score,
  r.`cb_event_score` as event_score,
  r.`trx_score` as trx_score,
  r.`telco_score` as telco_score,
  r.`ca_cic_score` as cic_score,
  r.`cash_alpha_stack_score`,
  case when date(ln_appln_submit_datetime) >= '2026-01-01' and date(ln_appln_submit_datetime) <= '2026-04-30' then 'Dev_Train'
           when date(ln_appln_submit_datetime) > '2026-04-30' and  date(ln_appln_submit_datetime) < '2026-10-30'then 'Dev_Test'
           when date(ln_appln_submit_datetime) < '2026-01-01' then 'Pre_Train'
           end Data_selection
FROM
  `prj-prod-dataplatform.worktable_data_analysis.cash_alpha_v2_combo_1_full_lr_applied_backscore_20260101_20261027` r
where r.cash_alpha_stack_score is not null  
and trench_category = 'Trench 1'
and date(r.ln_appln_submit_datetime) < '2026-10-30' 
;
"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID 07c43665-7b03-4ed3-b358-9a69b8dfa9b1 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (60575, 18)


In [14]:
feature_column = ['demo_score','app_score', 'device_score', 'credo_score', 'event_score', 'trx_score', 'telco_score', 'cic_score', 'cash_alpha_stack_score']

dfd = transform_datav2(data, feature_column, a='cash_alpha_stack_score', modelDisplayName='Alpha-Cash-Stack-Model', tc='Trench 1', subscription_name = 'Cash Stack v2 Models') 
dfd.head()

,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,4460633,5489e256-0c9c-4d34-90c4-c8ea3d6171f0,e597f56c-809b-4f9d-a282-ca19ab1682b1,0.044513,2026-11-10T12:23:55.173839,2026-11-10T12:23:55.173839,Alpha-Cash-Stack-Model,v2,"{""demo_score"": 0.10978910624089352, ""app_score...",Cash Stack v2 Models,a9e429b5-d0e2-4c66-a161-36c108da04ef,2026-11-10T12:23:55.173839,{},Trench 1,Android,Dev_Test,2026-07-09 07:58:04
1,4116540,6742f7e5-55d4-4d01-9ed6-56e5aa4e7685,05a6fc7f-89c9-43d9-8282-99e912f43a3f,0.089506,2026-11-10T12:23:55.175032,2026-11-10T12:23:55.175032,Alpha-Cash-Stack-Model,v2,"{""demo_score"": 0.22002740541290447, ""app_score...",Cash Stack v2 Models,d3f42cea-25be-4a0b-a15a-a2c2d8f2c350,2026-11-10T12:23:55.175032,{},Trench 1,Android,Dev_Train,2026-03-10 15:16:59
2,4151340,fd34a2b7-dc82-4c46-a870-acdb32f94f37,af29e766-7219-4337-98bc-a3acddb08a94,0.046735,2026-11-10T12:23:55.175604,2026-11-10T12:23:55.175604,Alpha-Cash-Stack-Model,v2,"{""demo_score"": 0.08805854413243633, ""app_score...",Cash Stack v2 Models,ee67109a-7378-420c-979d-dd87b56a4736,2026-11-10T12:23:55.175604,{},Trench 1,Android,Dev_Train,2026-02-28 14:25:45
3,4452996,e7ac83da-71c1-4e4c-a3a4-bf5947812c26,60fa5804-1024-4368-a231-9cf8f98ffe9a,0.039600,2026-11-10T12:23:55.176137,2026-11-10T12:23:55.176137,Alpha-Cash-Stack-Model,v2,"{""demo_score"": 0.1091882070136843, ""app_score""...",Cash Stack v2 Models,d4da4244-7a6b-468c-86ff-ccb8d1345aa7,2026-11-10T12:23:55.176137,{},Trench 1,Android,Dev_Test,2026-07-05 19:12:22
4,4140075,a17e94f5-bf08-4d7a-92b1-2d48ff82aeaf,5dd74f63-4b12-4376-8074-0071f8ec6bc0,0.051496,2026-11-10T12:23:55.176725,2026-11-10T12:23:55.176725,Alpha-Cash-Stack-Model,v2,"{""demo_score"": 0.1342112554960421, ""app_score""...",Cash Stack v2 Models,20951524-d255-44ba-9c00-f3b4cd2c56bc,2026-11-10T12:23:55.176725,{},Trench 1,Android,Dev_Train,2026-02-25 19:13:31


In [16]:
result = dfd.groupby(['Data_selection', 'deviceOs', 'modelVersionId']).agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,deviceOs,modelVersionId,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,Android,v2,6498,2026-05-01 00:20:27,2026-10-27 18:19:33
1,Dev_Test,iOS,v2,6861,2026-05-01 06:29:58,2026-10-27 23:37:38
2,Dev_Train,Android,v2,34330,2026-01-01 00:54:14,2026-04-30 23:09:40
3,Dev_Train,iOS,v2,12886,2026-01-01 00:14:31,2026-04-30 23:25:54


In [17]:
# Upload to BigQuery
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=3a8f8c30-da91-4983-97b2-7e046d7ebf5d>

### ₸ Trench 2

#### ₫ Delete

In [18]:
sq = """
DELETE FROM `prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116`
WHERE modelDisplayName = 'Alpha-Cash-Stack-Model'
  AND modelVersionId = 'v2'
  AND trenchCategory = 'Trench 2';
"""

query_job = client.query(sq)
query_job.result()

print(f"Deleted {query_job.num_dml_affected_rows} rows.")

Deleted 0 rows.


In [19]:
sq = """
SELECT
  r.`customerId` as customer_id,
  r.`digitalLoanAccountId`,
  r.`ln_appln_submit_datetime` application_date,
  r.`ln_loan_type`,
  r.`ln_loan_level_user_type`,
  r.`trench_category`,
  r.`ln_os_type` as osType,
  r.`is_android`,
  r.`cb_demo_score` as demo_score,
  r.`apps_score` as app_score,
  r.`device_score` as device_score,
  r.`c_credo_score` as credo_score,
  r.`cb_event_score` as event_score,
  r.`trx_score` as trx_score,
  r.`telco_score` as telco_score,
  r.`ca_cic_score` as cic_score,
  r.`cash_alpha_stack_score`,
  case when date(ln_appln_submit_datetime) >= '2026-01-01' and date(ln_appln_submit_datetime) <= '2026-04-30' then 'Dev_Train'
           when date(ln_appln_submit_datetime) > '2026-04-30' and  date(ln_appln_submit_datetime) < '2026-10-30'then 'Dev_Test'
           when date(ln_appln_submit_datetime) < '2026-01-01' then 'Pre_Train'
           end Data_selection
FROM
  `prj-prod-dataplatform.worktable_data_analysis.cash_alpha_v2_combo_1_full_lr_applied_backscore_20260101_20261027` r
where r.cash_alpha_stack_score is not null  
and trench_category = 'Trench 2'
and date(r.ln_appln_submit_datetime) < '2026-10-30' 
;
"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID fc644a5d-d2cf-414e-8043-1aff738edd9e successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (36004, 18)


In [20]:
feature_column = ['demo_score','app_score', 'device_score', 'credo_score', 'event_score', 'trx_score', 'telco_score', 'cic_score', 'cash_alpha_stack_score']

dfd = transform_datav2(data, feature_column, a='cash_alpha_stack_score', modelDisplayName='Alpha-Cash-Stack-Model', tc='Trench 2', subscription_name = 'Cash Stack v2 Models') 
dfd.head()

,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,3904609,fa873fec-4b34-4628-a038-6fd0048db232,b5d249b4-aa20-4d0c-843e-9d6190512e69,0.033044,2026-11-10T12:27:02.299497,2026-11-10T12:27:02.299497,Alpha-Cash-Stack-Model,v2,"{""demo_score"": 0.17616778084380624, ""app_score...",Cash Stack v2 Models,e765a8f5-4a31-4f51-8ea5-3147b2bea9f2,2026-11-10T12:27:02.299497,{},Trench 2,Android,Dev_Test,2026-05-09 02:14:31
1,1289416,625a8d60-d592-4443-bc54-4efc6af34a58,92d0a4c2-5147-4db5-bee9-9a337ca5359d,0.032976,2026-11-10T12:27:02.300520,2026-11-10T12:27:02.300520,Alpha-Cash-Stack-Model,v2,"{""demo_score"": 0.3198099682850999, ""app_score""...",Cash Stack v2 Models,2cf69df3-8ee7-437c-b5d7-d4f781ef4cd3,2026-11-10T12:27:02.300520,{},Trench 2,Android,Dev_Test,2026-06-16 07:26:06
2,3326484,be86f6a3-cc0e-4723-8618-9c375fff0770,65e68f61-d23c-40cb-b660-7730be716331,0.070797,2026-11-10T12:27:02.300520,2026-11-10T12:27:02.300520,Alpha-Cash-Stack-Model,v2,"{""demo_score"": 0.23234375285850709, ""app_score...",Cash Stack v2 Models,352cde5c-bb0a-46ad-94be-e6b4616deb41,2026-11-10T12:27:02.300520,{},Trench 2,Android,Dev_Train,2026-02-27 17:03:37
3,4071224,8f829df3-ffad-414c-9355-b29faed620c6,97a61651-6301-44c0-bc70-ac2235a28176,0.020052,2026-11-10T12:27:02.301518,2026-11-10T12:27:02.301518,Alpha-Cash-Stack-Model,v2,"{""demo_score"": 0.0906047851085707, ""app_score""...",Cash Stack v2 Models,80662e67-3078-4d18-89c3-870736fb6ff4,2026-11-10T12:27:02.301518,{},Trench 2,Android,Dev_Test,2026-09-21 20:55:11
4,3896536,c1cdf5bd-e3e9-4514-9543-eff3f80805f0,38cd57fd-58aa-42d1-b302-1d9b522dfedc,0.017714,2026-11-10T12:27:02.301518,2026-11-10T12:27:02.301518,Alpha-Cash-Stack-Model,v2,"{""demo_score"": 0.15116630477955423, ""app_score...",Cash Stack v2 Models,526c53fb-b34e-40b2-be0e-3dff492cce7a,2026-11-10T12:27:02.301518,{},Trench 2,Android,Dev_Train,2026-01-15 15:06:07


In [21]:
result = dfd.groupby(['Data_selection', 'deviceOs', 'modelVersionId', 'trenchCategory']).agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,deviceOs,modelVersionId,trenchCategory,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,Android,v2,Trench 2,6117,2026-05-01 00:12:54,2026-10-27 23:48:11
1,Dev_Test,iOS,v2,Trench 2,3592,2026-05-01 01:43:33,2026-10-27 23:36:03
2,Dev_Train,Android,v2,Trench 2,17031,2026-01-01 01:24:57,2026-04-30 23:18:18
3,Dev_Train,iOS,v2,Trench 2,9264,2026-01-01 08:51:47,2026-04-30 22:10:58


In [22]:
# Upload to BigQuery
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=bcf2f3f3-15d1-49ab-a0a4-9915d9e9d9e0>

### ₸ Trench 3

#### ₫ Delete

In [23]:
sq = """
DELETE FROM `prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116`
WHERE modelDisplayName = 'Alpha-Cash-Stack-Model'
  AND modelVersionId = 'v2'
  AND trenchCategory = 'Trench 3';
"""

query_job = client.query(sq)
query_job.result()

print(f"Deleted {query_job.num_dml_affected_rows} rows.")

Deleted 0 rows.


In [24]:
sq = """
SELECT
  r.`customerId` as customer_id,
  r.`digitalLoanAccountId`,
  r.`ln_appln_submit_datetime` application_date,
  r.`ln_loan_type`,
  r.`ln_loan_level_user_type`,
  r.`trench_category`,
  r.`ln_os_type` as osType,
  r.`is_android`,
  r.`cb_demo_score` as demo_score,
  r.`apps_score` as app_score,
  r.`device_score` as device_score,
  r.`c_credo_score` as credo_score,
  r.`cb_event_score` as event_score,
  r.`trx_score` as trx_score,
  r.`telco_score` as telco_score,
  r.`ca_cic_score` as cic_score,
  r.`cash_alpha_stack_score`,
  case when date(ln_appln_submit_datetime) >= '2026-01-01' and date(ln_appln_submit_datetime) <= '2026-04-30' then 'Dev_Train'
           when date(ln_appln_submit_datetime) > '2026-04-30' and  date(ln_appln_submit_datetime) < '2026-10-30'then 'Dev_Test'
           when date(ln_appln_submit_datetime) < '2026-01-01' then 'Pre_Train'
           end Data_selection
FROM
  `prj-prod-dataplatform.worktable_data_analysis.cash_alpha_v2_combo_1_full_lr_applied_backscore_20260101_20261027` r
where r.cash_alpha_stack_score is not null  
and trench_category = 'Trench 3'
and date(r.ln_appln_submit_datetime) < '2026-10-30' 
;
"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID 57a560e1-1241-40b8-9b01-93490e4f5528 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (19091, 18)


In [25]:
feature_column = ['demo_score','app_score', 'device_score', 'credo_score', 'event_score', 'trx_score', 'telco_score', 'cic_score', 'cash_alpha_stack_score']

dfd = transform_datav2(data, feature_column, a='cash_alpha_stack_score', modelDisplayName='Alpha-Cash-Stack-Model', tc='Trench 3', subscription_name = 'Cash Stack v2 Models') 
dfd.head()

,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,3496861,01c3f3c3-e10b-4512-9f78-91a6a81d28c6,73e192b7-90c5-43a6-9dfd-a832e88d095c,0.190028,2026-11-10T12:28:27.132709,2026-11-10T12:28:27.132709,Alpha-Cash-Stack-Model,v2,"{""demo_score"": 0.27878973705630516, ""app_score...",Cash Stack v2 Models,6d5f7499-ea2e-4ca3-888d-8d9140813b67,2026-11-10T12:28:27.132709,{},Trench 3,Android,Dev_Train,2026-03-27 08:24:10
1,4128809,c976f8f5-3748-4fb7-8c35-3b0e19d64084,d76c16ea-bc61-4b46-8add-0abed39280f1,0.129764,2026-11-10T12:28:27.133710,2026-11-10T12:28:27.133710,Alpha-Cash-Stack-Model,v2,"{""demo_score"": 0.1301823793152878, ""app_score""...",Cash Stack v2 Models,0f3c93e3-b124-4543-adbe-a7064d8466a1,2026-11-10T12:28:27.133710,{},Trench 3,Android,Dev_Test,2026-09-17 13:59:38
2,4038440,d86938b7-f1a9-4246-8c16-8016833645db,d5595f62-8797-440f-a837-95381b1321ea,0.068287,2026-11-10T12:28:27.134714,2026-11-10T12:28:27.134714,Alpha-Cash-Stack-Model,v2,"{""demo_score"": 0.11364539391405648, ""app_score...",Cash Stack v2 Models,6fb13df9-42f2-4e4c-918e-7e5a5f5f7860,2026-11-10T12:28:27.134714,{},Trench 3,Android,Dev_Test,2026-06-27 12:08:48
3,2445679,d9001cc6-9e47-4009-a985-eeb5e9d66862,ba63e93a-b4f5-454f-b364-eaf363348d4a,0.047895,2026-11-10T12:28:27.135715,2026-11-10T12:28:27.135715,Alpha-Cash-Stack-Model,v2,"{""demo_score"": 0.10618631895412108, ""app_score...",Cash Stack v2 Models,8b7ee748-c179-480f-88d9-339605f5b322,2026-11-10T12:28:27.135715,{},Trench 3,Android,Dev_Train,2026-01-12 10:00:49
4,1321424,ed8f2e67-8d6b-4678-ad76-e287c39b88cf,47c446a8-3c37-4285-ab71-21ca743d5f95,0.328329,2026-11-10T12:28:27.136236,2026-11-10T12:28:27.136236,Alpha-Cash-Stack-Model,v2,"{""demo_score"": 0.12366107155096254, ""app_score...",Cash Stack v2 Models,f9206d75-717d-4df9-b0cf-052a0983a8e5,2026-11-10T12:28:27.136236,{},Trench 3,Android,Dev_Test,2026-05-18 15:13:07


In [26]:
result = dfd.groupby(['Data_selection', 'deviceOs', 'modelVersionId', 'trenchCategory']).agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,deviceOs,modelVersionId,trenchCategory,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,Android,v2,Trench 3,7453,2026-05-01 06:51:12,2026-10-27 19:26:07
1,Dev_Test,iOS,v2,Trench 3,4333,2026-05-01 00:25:51,2026-10-27 23:29:58
2,Dev_Train,Android,v2,Trench 3,4516,2026-01-01 09:25:56,2026-04-30 22:14:13
3,Dev_Train,iOS,v2,Trench 3,2789,2026-01-01 07:37:17,2026-04-30 22:48:10


In [27]:
# Upload to BigQuery
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=c1406ff2-4f08-44d4-bae7-c12f0eae4414>

## Ⓜ️ beta_stack_model_cash

### ₸ Trench 1

#### ₫ Delete

In [28]:
sq = """
DELETE FROM `prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116`
WHERE modelDisplayName = 'Beta-Cash-Stack-Model'
  AND modelVersionId = 'v2'
  AND trenchCategory = 'Trench 1';
"""

query_job = client.query(sq)
query_job.result()

print(f"Deleted {query_job.num_dml_affected_rows} rows.")

Deleted 0 rows.


In [ ]:
sq = """
select 
 `customerId` as customer_id,
  `digitalLoanAccountId`,
  `ln_appln_submit_datetime` as application_date,
  `ln_disb_dtime`,
  `ln_loan_type`,
  `ln_loan_level_user_type`,
  `trench_category`,
  `ln_os_type` as osType,
  `is_android`,
  `cb_demo_score` as demo_score,
  `apps_score` as app_score,
  `device_score` as device_score,
  `c_credo_score` as credo_score,
  `cb_event_score` as event_score,
  `trx_score` as trx_score,
  cash_beta_stack_score,
  case when date(ln_appln_submit_datetime) >= '2026-01-01' and date(ln_appln_submit_datetime) <= '2026-04-30' then 'Dev_Train'
           when date(ln_appln_submit_datetime) > '2026-04-30' and  date(ln_appln_submit_datetime) < '2026-10-30'then 'Dev_Test'
           when date(ln_appln_submit_datetime) < '2026-01-01' then 'Pre_Train'
           end Data_selection
from `prj-prod-dataplatform.worktable_data_analysis.cash_beta_v2_combo_1_full_lr_applied_backscore_20260101_20261027` r
where r.cash_beta_stack_score is not null  
and trench_category = 'Trench 1'
and date(r.ln_appln_submit_datetime) < '2026-10-30' 
;
"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID 0442df43-d0c1-48d1-98f4-766272b8bec3 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (373700, 17)


In [30]:
feature_column = ['demo_score','app_score', 'device_score', 'credo_score', 'event_score', 'trx_score',  'cash_beta_stack_score']

dfd = transform_datav2(data, feature_column, a='cash_beta_stack_score', modelDisplayName='Beta-Cash-Stack-Model', tc='Trench 1', subscription_name = 'Cash Stack v2 Models') 
dfd.head()

,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,4460633,5489e256-0c9c-4d34-90c4-c8ea3d6171f0,e1cf3fc0-c2f3-4f21-81eb-ef4549723652,0.053481,2026-11-10T12:40:39.552362,2026-11-10T12:40:39.552362,Beta-Cash-Stack-Model,v2,"{""cash_beta_stack_score"": 0.053480852323276}",Cash Stack v2 Models,0543a9a4-1db0-4739-a9c8-88faf74cf3ea,2026-11-10T12:40:39.552362,{},Trench 1,Android,Dev_Test,2026-07-09 07:58:04
1,4116540,6742f7e5-55d4-4d01-9ed6-56e5aa4e7685,aaa86412-0b7e-4a33-a233-4ea987a1fb33,0.056003,2026-11-10T12:40:39.552362,2026-11-10T12:40:39.552362,Beta-Cash-Stack-Model,v2,"{""cash_beta_stack_score"": 0.05600309051251875}",Cash Stack v2 Models,cca0a04f-184b-469b-bfa0-cf980d54e38f,2026-11-10T12:40:39.552362,{},Trench 1,Android,Dev_Train,2026-03-10 15:16:59
2,4256218,83dd42fb-7858-46bf-830f-e71dba7c85cf,b3ccdd00-fdd6-4594-931e-636b5d660047,0.039947,2026-11-10T12:40:39.554362,2026-11-10T12:40:39.554362,Beta-Cash-Stack-Model,v2,"{""cash_beta_stack_score"": 0.03994716828236957}",Cash Stack v2 Models,87eb56a6-cae7-459a-aaca-310ac280b5f0,2026-11-10T12:40:39.554362,{},Trench 1,Android,Dev_Train,2026-04-02 15:43:28
3,4034184,ca74c318-3224-4431-93cb-35134b8f42fa,2a191e66-45f8-4c0d-bcd8-8f9b1896e0b9,0.074731,2026-11-10T12:40:39.554362,2026-11-10T12:40:39.554362,Beta-Cash-Stack-Model,v2,"{""cash_beta_stack_score"": 0.07473142939299604}",Cash Stack v2 Models,65111de8-fcc2-4582-b158-d8cc8fedcfbe,2026-11-10T12:40:39.554362,{},Trench 1,Android,Dev_Train,2026-01-23 01:31:04
4,4144091,56c99fdc-0f90-4b16-b268-1682592383b7,680564ea-2b25-485b-ae0a-3a871ded9ac0,0.118964,2026-11-10T12:40:39.554362,2026-11-10T12:40:39.554362,Beta-Cash-Stack-Model,v2,"{""cash_beta_stack_score"": 0.11896443973125323}",Cash Stack v2 Models,cb0631c8-21b3-46a2-bb23-e2dd02f6e6f3,2026-11-10T12:40:39.554362,{},Trench 1,Android,Dev_Train,2026-03-01 19:11:29


In [31]:
result = dfd.groupby(['Data_selection', 'deviceOs', 'modelVersionId']).agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,deviceOs,modelVersionId,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,Android,v2,132405,2026-05-01 00:09:44,2026-10-27 21:53:21
1,Dev_Test,iOS,v2,59434,2026-05-01 00:00:16,2026-10-27 23:37:38
2,Dev_Train,Android,v2,147463,2026-01-01 00:16:07,2026-04-30 23:59:24
3,Dev_Train,iOS,v2,34398,2026-01-01 00:14:31,2026-04-30 23:53:26


In [32]:
# Upload to BigQuery
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=29ac7deb-141d-49b1-9f3b-998a72e8bfff>

### ₸ Trench 2

#### ₫ Delete

In [33]:
sq = """
DELETE FROM `prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116`
WHERE modelDisplayName = 'Beta-Cash-Stack-Model'
  AND modelVersionId = 'v2'
  AND trenchCategory = 'Trench 2';
"""

query_job = client.query(sq)
query_job.result()

print(f"Deleted {query_job.num_dml_affected_rows} rows.")

Deleted 0 rows.


In [34]:
sq = """
select 
 `customerId` as customer_id,
  `digitalLoanAccountId`,
  `ln_appln_submit_datetime` as application_date,
  `ln_disb_dtime`,
  `ln_loan_type`,
  `ln_loan_level_user_type`,
  `trench_category`,
  `ln_os_type` as osType,
  `is_android`,
  `cb_demo_score` as demo_score,
  `apps_score` as app_score,
  `device_score` as device_score,
  `c_credo_score` as credo_score,
  `cb_event_score` as event_score,
  `trx_score` as trx_score,
  cash_beta_stack_score,
  case when date(ln_appln_submit_datetime) >= '2026-01-01' and date(ln_appln_submit_datetime) <= '2026-04-30' then 'Dev_Train'
           when date(ln_appln_submit_datetime) > '2026-04-30' and  date(ln_appln_submit_datetime) < '2026-10-30'then 'Dev_Test'
           when date(ln_appln_submit_datetime) < '2026-01-01' then 'Pre_Train'
           end Data_selection
from `prj-prod-dataplatform.worktable_data_analysis.cash_beta_v2_combo_1_full_lr_applied_backscore_20260101_20261027` r
where r.cash_beta_stack_score is not null  
and trench_category = 'Trench 2'
and date(r.ln_appln_submit_datetime) < '2026-10-30' 
;
"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID 6f81ca54-9832-45c5-92a8-5fe3ebeb9a73 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (206696, 17)


In [35]:
feature_column = ['demo_score','app_score', 'device_score', 'credo_score', 'event_score', 'trx_score',  'cash_beta_stack_score']

dfd = transform_datav2(data, feature_column, a='cash_beta_stack_score', modelDisplayName='Beta-Cash-Stack-Model', tc='Trench 2', subscription_name = 'Cash Stack v2 Models') 
dfd.head()

,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,3904609,fa873fec-4b34-4628-a038-6fd0048db232,92b6b8be-a60c-40da-a8e6-0ae10d66c859,0.026742,2026-11-10T12:43:37.605624,2026-11-10T12:43:37.605624,Beta-Cash-Stack-Model,v2,"{""demo_score"": 0.17616778084380624, ""app_score...",Cash Stack v2 Models,70b98f07-0613-4381-817e-bcd9a1ae470c,2026-11-10T12:43:37.605624,{},Trench 2,Android,Dev_Test,2026-05-09 02:14:31
1,1289416,625a8d60-d592-4443-bc54-4efc6af34a58,159b0c10-7875-448b-bbbf-e54ae3eb1f00,0.019981,2026-11-10T12:43:37.606650,2026-11-10T12:43:37.606650,Beta-Cash-Stack-Model,v2,"{""demo_score"": 0.3198099682850999, ""app_score""...",Cash Stack v2 Models,d586a8b8-b10d-41ff-9e9f-cfd00889bbb2,2026-11-10T12:43:37.606650,{},Trench 2,Android,Dev_Test,2026-06-16 07:26:06
2,3326484,be86f6a3-cc0e-4723-8618-9c375fff0770,6f5e6843-6061-45a6-9ab1-24b2439c4acb,0.094449,2026-11-10T12:43:37.607671,2026-11-10T12:43:37.607671,Beta-Cash-Stack-Model,v2,"{""demo_score"": 0.23234375285850709, ""app_score...",Cash Stack v2 Models,9a10c441-3f0d-4831-929c-ff3f57c99e6e,2026-11-10T12:43:37.607671,{},Trench 2,Android,Dev_Train,2026-02-27 17:03:37
3,4071224,8f829df3-ffad-414c-9355-b29faed620c6,a3d915b4-de11-49f4-8cfd-b67c4a39b904,0.021857,2026-11-10T12:43:37.607671,2026-11-10T12:43:37.607671,Beta-Cash-Stack-Model,v2,"{""demo_score"": 0.0906047851085707, ""app_score""...",Cash Stack v2 Models,52256dec-7569-4f62-a125-19b15c1fb845,2026-11-10T12:43:37.607671,{},Trench 2,Android,Dev_Test,2026-09-21 20:55:11
4,3896536,c1cdf5bd-e3e9-4514-9543-eff3f80805f0,a2278a75-d890-44bf-a009-9a92030f6f73,0.030279,2026-11-10T12:43:37.608668,2026-11-10T12:43:37.608668,Beta-Cash-Stack-Model,v2,"{""demo_score"": 0.15116630477955423, ""app_score...",Cash Stack v2 Models,7a46c65c-cf34-41d8-af90-630aca1fbed5,2026-11-10T12:43:37.608668,{},Trench 2,Android,Dev_Train,2026-01-15 15:06:07


In [36]:
result = dfd.groupby(['Data_selection', 'deviceOs', 'modelVersionId', 'trenchCategory']).agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,deviceOs,modelVersionId,trenchCategory,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,Android,v2,Trench 2,85884,2026-05-01 00:02:43,2026-10-27 23:52:37
1,Dev_Test,iOS,v2,Trench 2,39588,2026-05-01 01:03:45,2026-10-27 23:53:29
2,Dev_Train,Android,v2,Trench 2,56711,2026-01-01 00:18:41,2026-04-30 23:58:24
3,Dev_Train,iOS,v2,Trench 2,24513,2026-01-01 00:20:21,2026-04-30 23:59:25


In [37]:
# Upload to BigQuery
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=c7a08f34-2dbd-4f99-a4f1-47209d3715f9>

### ₸ Trench 3

#### ₫ Delete

In [38]:
sq = """
DELETE FROM `prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116`
WHERE modelDisplayName = 'Beta-Cash-Stack-Model'
  AND modelVersionId = 'v2'
  AND trenchCategory = 'Trench 3';
"""

query_job = client.query(sq)
query_job.result()

print(f"Deleted {query_job.num_dml_affected_rows} rows.")

Deleted 0 rows.


In [39]:
sq = """
select 
 `customerId` as customer_id,
  `digitalLoanAccountId`,
  `ln_appln_submit_datetime` as application_date,
  `ln_disb_dtime`,
  `ln_loan_type`,
  `ln_loan_level_user_type`,
  `trench_category`,
  `ln_os_type` as osType,
  `is_android`,
  `cb_demo_score` as demo_score,
  `apps_score` as app_score,
  `device_score` as device_score,
  `c_credo_score` as credo_score,
  `cb_event_score` as event_score,
  `trx_score` as trx_score,
  cash_beta_stack_score,
  case when date(ln_appln_submit_datetime) >= '2026-01-01' and date(ln_appln_submit_datetime) <= '2026-04-30' then 'Dev_Train'
           when date(ln_appln_submit_datetime) > '2026-04-30' and  date(ln_appln_submit_datetime) < '2026-10-30'then 'Dev_Test'
           when date(ln_appln_submit_datetime) < '2026-01-01' then 'Pre_Train'
           end Data_selection
from `prj-prod-dataplatform.worktable_data_analysis.cash_beta_v2_combo_1_full_lr_applied_backscore_20260101_20261027` r
where r.cash_beta_stack_score is not null  
and trench_category = 'Trench 3'
and date(r.ln_appln_submit_datetime) < '2026-10-30' 
;
"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

Job ID fafff4fc-f9b6-407a-bcaf-01a1dc292fb3 successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The shape of the dataframe is:	 (39631, 17)


In [40]:
feature_column = ['demo_score','app_score', 'device_score', 'credo_score', 'event_score', 'trx_score',  'cash_beta_stack_score']

dfd = transform_datav2(data, feature_column, a='cash_beta_stack_score', modelDisplayName='Beta-Cash-Stack-Model', tc='Trench 3', subscription_name = 'Cash Stack v2 Models') 
dfd.head()

,customerId,digitalLoanAccountId,crifApplicationId,prediction,start_time,end_time,modelDisplayName,modelVersionId,calcFeature,subscription_name,message_id,publish_time,attributes,trenchCategory,deviceOs,Data_selection,Application_date
0,3496861,01c3f3c3-e10b-4512-9f78-91a6a81d28c6,9c9341a6-8680-4eb8-a96f-00c57e6c7f4c,0.175335,2026-11-10T12:45:30.822899,2026-11-10T12:45:30.822899,Beta-Cash-Stack-Model,v2,"{""demo_score"": 0.27878973705630516, ""app_score...",Cash Stack v2 Models,ad10ac5d-a08c-40b5-a396-18b88658d930,2026-11-10T12:45:30.822899,{},Trench 3,Android,Dev_Train,2026-03-27 08:24:10
1,4128809,c976f8f5-3748-4fb7-8c35-3b0e19d64084,81dc5088-f2d5-4140-b32a-2444d758b1cf,0.095924,2026-11-10T12:45:30.823896,2026-11-10T12:45:30.823896,Beta-Cash-Stack-Model,v2,"{""demo_score"": 0.1301823793152878, ""app_score""...",Cash Stack v2 Models,42682191-8778-4aea-8857-5cbdd3d2dbd6,2026-11-10T12:45:30.823896,{},Trench 3,Android,Dev_Test,2026-09-17 13:59:38
2,4038440,d86938b7-f1a9-4246-8c16-8016833645db,3c44de42-491a-45f4-8c93-da664eb6854a,0.049989,2026-11-10T12:45:30.823896,2026-11-10T12:45:30.823896,Beta-Cash-Stack-Model,v2,"{""demo_score"": 0.11364539391405648, ""app_score...",Cash Stack v2 Models,ea3e830a-5c0e-4017-b68f-b6f39febfeb7,2026-11-10T12:45:30.823896,{},Trench 3,Android,Dev_Test,2026-06-27 12:08:48
3,1103603,0ce974ee-bd51-4313-9386-78085881b46a,4e67c2c7-faef-42a7-ba9e-14b6f3320b87,0.045264,2026-11-10T12:45:30.824896,2026-11-10T12:45:30.824896,Beta-Cash-Stack-Model,v2,"{""demo_score"": 0.13359573303435293, ""app_score...",Cash Stack v2 Models,b6e35562-6a44-4db7-a115-eda2d6948db9,2026-11-10T12:45:30.824896,{},Trench 3,Android,Dev_Train,2026-04-26 08:58:43
4,2445679,d9001cc6-9e47-4009-a985-eeb5e9d66862,40bc6575-cd4b-496d-8a64-77183e658fb5,0.053373,2026-11-10T12:45:30.824896,2026-11-10T12:45:30.824896,Beta-Cash-Stack-Model,v2,"{""demo_score"": 0.10618631895412108, ""app_score...",Cash Stack v2 Models,4d3fe244-6451-4188-94ce-7e10d5a0cd40,2026-11-10T12:45:30.824896,{},Trench 3,Android,Dev_Train,2026-01-12 10:00:49


In [41]:
result = dfd.groupby(['Data_selection', 'deviceOs', 'modelVersionId', 'trenchCategory']).agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

,Data_selection,deviceOs,modelVersionId,trenchCategory,digitalLoanAccountId_count,Application_date_min,Application_date_max
0,Dev_Test,Android,v2,Trench 3,18266,2026-05-01 00:18:54,2026-10-27 23:34:58
1,Dev_Test,iOS,v2,Trench 3,8577,2026-05-01 00:25:51,2026-10-27 23:29:58
2,Dev_Train,Android,v2,Trench 3,8279,2026-01-01 01:54:04,2026-04-30 23:53:09
3,Dev_Train,iOS,v2,Trench 3,4509,2026-01-01 00:34:08,2026-04-30 22:48:10


In [42]:
# Upload to BigQuery
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=a1077554-022c-4d80-84f6-7c50cda316c9>

# 🪦💀 Graveyard

#### PSI Functions new

In [ ]:
# ## Updated on 27-10-2025 - Modified for Training Period Baseline
# import pandas as pd
# import numpy as np
# from typing import List, Dict, Tuple
# import warnings
# warnings.filterwarnings('ignore')

# def identify_feature_types(df: pd.DataFrame, feature_list: List[str]) -> Dict[str, List[str]]:
#     """
#     Identify categorical and numerical features from the feature list.

#     Parameters:
#     -----------
#     df : pd.DataFrame
#         Input dataframe
#     feature_list : List[str]
#         List of features to classify

#     Returns:
#     --------
#     Dict with 'categorical' and 'numerical' keys containing respective feature lists
#     """
#     categorical_features = []
#     numerical_features = []

#     for feature in feature_list:
#         if feature not in df.columns:
#             print(f"Warning: Feature '{feature}' not found in dataframe")
#             continue

#         # Check if feature is numeric
#         if pd.api.types.is_numeric_dtype(df[feature]):
#             # If unique values are less than 15 and all integers, treat as categorical
#             unique_vals = df[feature].nunique()
#             if unique_vals < 15 and df[feature].dropna().apply(lambda x: x == int(x) if isinstance(x, (int, float)) else False).all():
#                 categorical_features.append(feature)
#             else:
#                 numerical_features.append(feature)
#         else:
#             categorical_features.append(feature)

#     return {
#         'categorical': categorical_features,
#         'numerical': numerical_features
#     }


# def create_bins_for_features(df: pd.DataFrame,
#                              numerical_features: List[str],
#                              categorical_features: List[str],
#                              train_period_df: pd.DataFrame) -> Dict:
#     """
#     Create bins for numerical features (deciles with fallback) and categorical features (top 6 + others)
#     based on the entire training period data.

#     Parameters:
#     -----------
#     df : pd.DataFrame
#         Full input dataframe
#     numerical_features : List[str]
#         List of numerical features
#     categorical_features : List[str]
#         List of categorical features
#     train_period_df : pd.DataFrame
#         Training period dataframe (June 2024 to March 2025)

#     Returns:
#     --------
#     Dictionary containing binning information for each feature
#     """
#     binning_info = {}

#     # Create bins for numerical features with fallback strategy
#     for feature in numerical_features:
#         valid_data = train_period_df[feature].dropna()

#         if len(valid_data) == 0:
#             binning_info[feature] = {'type': 'numerical', 'bins': None, 'bin_ranges': {}}
#             continue

#         bins = None
#         bin_count = None

#         # Try 10 bins (deciles)
#         try:
#             test_bins = np.percentile(valid_data, np.arange(0, 101, 10))
#             test_bins = np.unique(test_bins)
#             if len(test_bins) >= 11:  # 11 edges = 10 bins
#                 bins = test_bins
#                 bin_count = 10
#         except Exception as e:
#             pass

#         # If 10 bins not possible, try 5 bins
#         if bins is None:
#             try:
#                 test_bins = np.percentile(valid_data, np.arange(0, 101, 20))
#                 test_bins = np.unique(test_bins)
#                 if len(test_bins) >= 6:  # 6 edges = 5 bins
#                     bins = test_bins
#                     bin_count = 5
#             except Exception as e:
#                 pass

#         # If 5 bins not possible, try 3 bins
#         if bins is None:
#             try:
#                 test_bins = np.percentile(valid_data, [0, 33.33, 66.67, 100])
#                 test_bins = np.unique(test_bins)
#                 if len(test_bins) >= 4:  # 4 edges = 3 bins
#                     bins = test_bins
#                     bin_count = 3
#             except Exception as e:
#                 pass

#         # If still no bins possible, use equal distance bins of 5
#         if bins is None:
#             print(f"Warning: Feature '{feature}' has insufficient variance - cannot create standard bins")
#             print(f"Feature '{feature}': Using equal distance bins of 5")

#             min_val = valid_data.min()
#             max_val = valid_data.max()

#             # Create 5 equal distance bins
#             bins = np.linspace(min_val, max_val, 6)  # 6 edges = 5 bins
#             bins = np.unique(bins)
#             bin_count = len(bins) - 1

#             # If all values are the same, add slight buffer
#             if bin_count == 1:
#                 bins = np.array([min_val - 0.1, min_val, min_val + 0.1])
#                 bin_count = 2
#                 print(f"Feature '{feature}': Constant value ({min_val}). Created 2 equal distance bins with buffer")

#         # Add infinity edges to capture all values
#         bins = bins.copy()
#         bins[0] = -np.inf
#         bins[-1] = np.inf

#         print(f"Feature '{feature}': Created {bin_count} bins")

#         # Create bin ranges dictionary
#         bin_ranges = {}
#         for i in range(len(bins)-1):
#             bin_name = f"Bin_{i+1}"
#             bin_ranges[bin_name] = {
#                 'min': bins[i],
#                 'max': bins[i+1],
#                 'range_str': f"[{bins[i]:.2f}, {bins[i+1]:.2f}]" if not np.isinf(bins[i]) and not np.isinf(bins[i+1]) else f"({bins[i]}, {bins[i+1]})"
#             }

#         binning_info[feature] = {
#             'type': 'numerical',
#             'bins': bins,
#             'bin_ranges': bin_ranges,
#             'bin_count': bin_count
#         }

#     # Create bins for categorical features (top 6 + others) using training period
#     for feature in categorical_features:
#         value_counts = train_period_df[feature].value_counts()
#         unique_categories = value_counts.index.tolist()
#         print(f"Unique categories: {unique_categories}")

#         if len(unique_categories) <= 6:
#             # Treat each category as a separate bin
#             top_categories = unique_categories
#         else:
#             # Use top 6 categories only
#             top_categories = value_counts.nlargest(6).index.tolist()

#         print(f"Top categories for feature '{feature}': {top_categories}")

#         binning_info[feature] = {
#                 'type': 'categorical',
#                 'top_categories': top_categories,
#                 'bin_ranges': {}  # No ranges for categorical
#             }

#     return binning_info


# def apply_binning(df: pd.DataFrame,
#                   feature: str,
#                   binning_info: Dict) -> pd.Series:
#     """
#     Apply binning to a feature based on binning information.

#     Parameters:
#     -----------
#     df : pd.DataFrame
#         Input dataframe
#     feature : str
#         Feature name
#     binning_info : Dict
#         Binning information for the feature

#     Returns:
#     --------
#     pd.Series with binned values
#     """
#     if binning_info['type'] == 'numerical':
#         if binning_info['bins'] is None:
#             return pd.Series(['Missing'] * len(df), index=df.index)

#         bins = binning_info['bins']
#         labels = [f"Bin_{i+1}" for i in range(len(bins)-1)]

#         binned = pd.cut(df[feature],
#                        bins=bins,
#                        labels=labels,
#                        include_lowest=True,
#                        duplicates='drop')

#         # Handle nulls - convert to string and then replace
#         binned = binned.astype(str)
#         binned[df[feature].isna()] = 'Missing'

#         return binned

#     else:  # categorical
#         top_cats = binning_info['top_categories']

#         # Convert to string for consistent comparison
#         if pd.api.types.is_categorical_dtype(df[feature]):
#             feature_data = df[feature].astype(str)
#         else:
#             feature_data = df[feature].astype(str)

#         # Replace NaN string representation with 'Missing'
#         feature_data = feature_data.replace('nan', 'Missing')

#         # Convert top_cats to strings for comparison
#         top_cats_str = [str(cat) for cat in top_cats]

#         # Apply binning logic: use category name if in top_cats, else 'Others' (except for Missing)
#         binned = feature_data.apply(lambda x: x if x in top_cats_str else ('Others' if x != 'Missing' else 'Missing'))

#         return binned


# def calculate_psi(expected_pct: pd.Series,
#                   actual_pct: pd.Series,
#                   epsilon: float = 0.0001) -> float:
#     """
#     Calculate Population Stability Index with proper epsilon handling and renormalization.

#     Parameters:
#     -----------
#     expected_pct : pd.Series
#         Expected (baseline) percentages
#     actual_pct : pd.Series
#         Actual percentages
#     epsilon : float
#         Small value to avoid log(0)

#     Returns:
#     --------
#     PSI value
#     """
#     # Align indices
#     all_bins = expected_pct.index.union(actual_pct.index)
#     expected_pct = expected_pct.reindex(all_bins, fill_value=0)
#     actual_pct = actual_pct.reindex(all_bins, fill_value=0)

#     # Only add epsilon where values are zero
#     expected_pct = expected_pct.apply(lambda x: epsilon if x == 0 else x)
#     actual_pct = actual_pct.apply(lambda x: epsilon if x == 0 else x)

#     # Renormalize to ensure they sum to 1 after adding epsilon
#     expected_pct = expected_pct / expected_pct.sum()
#     actual_pct = actual_pct / actual_pct.sum()

#     # Calculate PSI
#     psi_value = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))

#     return psi_value


# def calculate_month_on_month_psi(df: pd.DataFrame,
#                                  feature_list: List[str],
#                                  segment_columns: List[str],
#                                  month_col: str = 'Application_month',
#                                  data_selection_col: str = 'Data_selection',
#                                  account_id_col: str = 'digitalLoanAccountId') -> pd.DataFrame:
#     """
#     Calculate PSI for each feature comparing training period (June 2024 to March 2025)
#     vs each month after March 2025, overall and by segments.

#     Parameters:
#     -----------
#     df : pd.DataFrame
#         Input dataframe
#     feature_list : List[str]
#         List of features to calculate PSI for
#     segment_columns : List[str]
#         List of segment columns
#     month_col : str
#         Name of month column
#     data_selection_col : str
#         Name of data selection column (identifies train period)
#     account_id_col : str
#         Name of account ID column for counting distinct accounts

#     Returns:
#     --------
#     pd.DataFrame with PSI values with one row per feature-month-segment combination
#     """
#     # Create a copy to avoid modifying original
#     df = df.copy()

#     # Identify training and test periods
#     train_df = df[df[data_selection_col] == 'Train'].copy()
#     test_df = df[df[data_selection_col] != 'Train'].copy()

#     if len(train_df) == 0:
#         raise ValueError("No training data found. Check Data_selection column.")

#     print(f"Training period: {train_df[month_col].min()} to {train_df[month_col].max()}")
#     print(f"Test period: {test_df[month_col].min()} to {test_df[month_col].max()}")

#     # Identify feature types
#     feature_types = identify_feature_types(df, feature_list)

#     # Create binning strategy based on training period
#     binning_info = create_bins_for_features(
#         df,
#         feature_types['numerical'],
#         feature_types['categorical'],
#         train_df
#     )

#     # Get sorted test months
#     test_months = sorted(test_df[month_col].unique())

#     results = []

#     # Calculate overall PSI
#     for feature in feature_list:
#         if feature not in df.columns:
#             continue

#         # Apply binning to entire dataset
#         df[f'{feature}_binned'] = apply_binning(df, feature, binning_info[feature])
#         # print(f"Feature binned {df[f'{feature}_binned']}")
#         # Get training period distribution (baseline)
#         train_baseline = df[df[data_selection_col] == 'Train'][f'{feature}_binned'].value_counts(normalize=True)

#         # Calculate PSI for each test month
#         for month in test_months:
#             actual_dist = df[df[month_col] == month][f'{feature}_binned'].value_counts(normalize=True)
#             psi_value = calculate_psi(train_baseline, actual_dist)

#             # Calculate average percentages across all bins
#             expected_avg_pct = train_baseline.mean() * 100
#             actual_avg_pct = actual_dist.mean() * 100

#             # # Count distinct accounts for segment
#             # base_segment_count = train_segment[account_id_col].nunique()
#             # actual_segment_count = actual_segment[account_id_col].nunique()


#             results.append({
#                 'Feature': feature,
#                 'Feature_Type': binning_info[feature]['type'],
#                 'Segment_Column': 'Overall',
#                 'Segment_Value': 'All',
#                 'Month': f"{month}",
#                 'Base_Month': 'Train (Jun 2024 - Mar 2025)',
#                 'Current_Month': month,
#                 'Expected_Percentage': expected_avg_pct,
#                 'Actual_Percentage': actual_avg_pct,
#                 'PSI': psi_value
#             })

#     # Calculate PSI by segments
#     for segment_col in segment_columns:
#         if segment_col not in df.columns:
#             continue

#         segments = df[segment_col].dropna().unique()

#         for segment_val in segments:
#             segment_df = df[df[segment_col] == segment_val]

#             for feature in feature_list:
#                 if feature not in df.columns:
#                     continue

#                 # Get training period distribution for segment
#                 train_segment = segment_df[segment_df[data_selection_col] == 'Train']
#                 if len(train_segment) == 0:
#                     continue

#                 train_baseline = train_segment[f'{feature}_binned'].value_counts(normalize=True)

#                 # Calculate PSI for each test month
#                 for month in test_months:
#                     actual_segment = segment_df[segment_df[month_col] == month]
#                     if len(actual_segment) == 0:
#                         continue

#                     actual_dist = actual_segment[f'{feature}_binned'].value_counts(normalize=True)
#                     psi_value = calculate_psi(train_baseline, actual_dist)

#                     # Calculate average percentages across all bins
#                     expected_avg_pct = train_baseline.mean() * 100
#                     actual_avg_pct = actual_dist.mean() * 100

#                     # Count distinct accounts for segment
#                     base_segment_count = train_segment[account_id_col].nunique()
#                     actual_segment_count = actual_segment[account_id_col].nunique()

#                     results.append({
#                         'Feature': feature,
#                         'Feature_Type': binning_info[feature]['type'],
#                         'Segment_Column': segment_col,
#                         'Segment_Value': segment_val,
#                         'Month': f"{month}",
#                         'Base_Month': 'Train (Jun 2024 - Mar 2025)',
#                         'Current_Month': month,
#                         'Base_Count': base_segment_count,
#                         'Actual_Count': actual_segment_count,
#                         'Expected_Percentage': expected_avg_pct,
#                         'Actual_Percentage': actual_avg_pct,
#                         'PSI': psi_value
#                     })

#     return pd.DataFrame(results)


# def calculate_bin_level_psi(df: pd.DataFrame,
#                             feature_list: List[str],
#                             segment_columns: List[str],
#                             month_col: str = 'Application_month',
#                             data_selection_col: str = 'Data_selection',
#                             account_id_col: str = 'digitalLoanAccountId') -> pd.DataFrame:
#     """
#     Calculate bin-level PSI for each feature comparing training period
#     vs each month after March 2025, overall and by segments.

#     Parameters:
#     -----------
#     df : pd.DataFrame
#         Input dataframe
#     feature_list : List[str]
#         List of features to calculate PSI for
#     segment_columns : List[str]
#         List of segment columns
#     month_col : str
#         Name of month column
#     data_selection_col : str
#         Name of data selection column
#     account_id_col : str
#         Name of account ID column for counting distinct accounts

#     Returns:
#     --------
#     pd.DataFrame with bin-level PSI details including bin ranges
#     """
#     # Create a copy to avoid modifying original
#     df = df.copy()

#     # Identify training and test periods
#     train_df = df[df[data_selection_col] == 'Train'].copy()
#     test_df = df[df[data_selection_col] != 'Train'].copy()

#     if len(train_df) == 0:
#         raise ValueError("No training data found. Check Data_selection column.")

#     print(f"Training period: {train_df[month_col].min()} to {train_df[month_col].max()}")
#     print(f"Test period: {test_df[month_col].min()} to {test_df[month_col].max()}")

#     # Identify feature types
#     feature_types = identify_feature_types(df, feature_list)

#     # Create binning strategy based on training period
#     binning_info = create_bins_for_features(
#         df,
#         feature_types['numerical'],
#         feature_types['categorical'],
#         train_df
#     )

#     # Get sorted test months
#     test_months = sorted(test_df[month_col].unique())

#     results = []
#     epsilon = 0.0001

#     # Calculate overall bin-level PSI
#     for feature in feature_list:
#         if feature not in df.columns:
#             continue

#         # Apply binning to entire dataset
#         df[f'{feature}_binned'] = apply_binning(df, feature, binning_info[feature])
#         # print(df[f'{feature}_binned'])

#         # Get training period distribution (baseline)
#         train_baseline = df[df[data_selection_col] == 'Train'][f'{feature}_binned'].value_counts(normalize=True)

#         # Calculate bin-level PSI for each test month
#         for month in test_months:
#             month_data = df[df[month_col] == month]
#             actual_dist = month_data[f'{feature}_binned'].value_counts(normalize=True)

#             # Count distinct accounts
#             base_count = df[df[data_selection_col] == 'Train'][account_id_col].nunique()
#             actual_count = month_data[account_id_col].nunique()

#             # Get all bins
#             all_bins = train_baseline.index.union(actual_dist.index)

#             for bin_name in all_bins:
#                 # Simplified epsilon logic - no redundancy
#                 expected_pct = train_baseline.get(bin_name, 0)
#                 actual_pct = actual_dist.get(bin_name, 0)

#                 # Add epsilon only if zero
#                 expected_pct = epsilon if expected_pct == 0 else expected_pct
#                 actual_pct = epsilon if actual_pct == 0 else actual_pct

#                 # Calculate bin-level PSI
#                 bin_psi = (actual_pct - expected_pct) * np.log(actual_pct / expected_pct)

#                 # Get bin range information
#                 bin_ranges = binning_info[feature]['bin_ranges']
#                 if bin_name in bin_ranges:
#                     bin_min = bin_ranges[bin_name]['min']
#                     bin_max = bin_ranges[bin_name]['max']
#                     bin_range = bin_ranges[bin_name]['range_str']
#                 else:
#                     # For categorical or special bins (Missing, Others)
#                     bin_min = None
#                     bin_max = None
#                     bin_range = bin_name

#                 results.append({
#                     'Feature': feature,
#                     'Feature_Type': binning_info[feature]['type'],
#                     'Segment_Column': 'Overall',
#                     'Segment_Value': 'All',
#                     'Month': f"{month}",
#                     'Base_Month': 'Train (Jun 2024 - Mar 2025)',
#                     'Current_Month': month,
#                     'Base_Count': base_count,
#                     'Actual_Count': actual_count,
#                     'Bin': bin_name,
#                     'Bin_Range': bin_range,
#                     'Bin_Min': bin_min,
#                     'Bin_Max': bin_max,
#                     'Base_Percentage': (train_baseline.get(bin_name, 0) * 100),
#                     'Actual_Percentage': (actual_dist.get(bin_name, 0) * 100),
#                     'Bin_PSI': bin_psi
#                 })

#     # Calculate bin-level PSI by segments
#     for segment_col in segment_columns:
#         if segment_col not in df.columns:
#             continue

#         segments = df[segment_col].dropna().unique()

#         for segment_val in segments:
#             segment_df = df[df[segment_col] == segment_val]

#             for feature in feature_list:
#                 if feature not in df.columns:
#                     continue

#                 # Get training period distribution for segment
#                 train_segment = segment_df[segment_df[data_selection_col] == 'Train']
#                 if len(train_segment) == 0:
#                     continue

#                 train_baseline = train_segment[f'{feature}_binned'].value_counts(normalize=True)

#                 # Calculate bin-level PSI for each test month
#                 for month in test_months:
#                     actual_segment = segment_df[segment_df[month_col] == month]
#                     if len(actual_segment) == 0:
#                         continue

#                     actual_dist = actual_segment[f'{feature}_binned'].value_counts(normalize=True)

#                     # Count distinct accounts for segment
#                     base_segment_count = train_segment[account_id_col].nunique()
#                     actual_segment_count = actual_segment[account_id_col].nunique()

#                     # Get all bins
#                     all_bins = train_baseline.index.union(actual_dist.index)

#                     for bin_name in all_bins:
#                         # Simplified epsilon logic - no redundancy
#                         expected_pct = train_baseline.get(bin_name, 0)
#                         actual_pct = actual_dist.get(bin_name, 0)

#                         # Add epsilon only if zero
#                         expected_pct = epsilon if expected_pct == 0 else expected_pct
#                         actual_pct = epsilon if actual_pct == 0 else actual_pct

#                         # Calculate bin-level PSI
#                         bin_psi = (actual_pct - expected_pct) * np.log(actual_pct / expected_pct)

#                         # Get bin range information
#                         bin_ranges = binning_info[feature]['bin_ranges']
#                         if bin_name in bin_ranges:
#                             bin_min = bin_ranges[bin_name]['min']
#                             bin_max = bin_ranges[bin_name]['max']
#                             bin_range = bin_ranges[bin_name]['range_str']
#                         else:
#                             # For categorical or special bins (Missing, Others)
#                             bin_min = None
#                             bin_max = None
#                             bin_range = bin_name

#                         results.append({
#                             'Feature': feature,
#                             'Feature_Type': binning_info[feature]['type'],
#                             'Segment_Column': segment_col,
#                             'Segment_Value': segment_val,
#                             'Month': f"{month}",
#                             'Base_Month': 'Train (Jun 2024 - Mar 2025)',
#                             'Current_Month': month,
#                             'Base_Count': base_segment_count,
#                             'Actual_Count': actual_segment_count,
#                             'Bin': bin_name,
#                             'Bin_Range': bin_range,
#                             'Bin_Min': bin_min,
#                             'Bin_Max': bin_max,
#                             'Base_Percentage': (train_baseline.get(bin_name, 0) * 100),
#                             'Actual_Percentage': (actual_dist.get(bin_name, 0) * 100),
#                             'Bin_PSI': bin_psi
#                         })

#     return pd.DataFrame(results)

# End